### **Part 1: Environment Setup and Library Imports**
This section initializes the necessary Python libraries and configures output directories for saving the final poster and individual high-resolution charts.


In [ ]:
import os
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import matplotlib.image as mpimg
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import qrcode

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

press_dir = r"C:\Users\USER\OneDrive\문서\press"
file_path = os.path.join(press_dir, "Infectious Disease Data.xlsx")
output_dir = os.path.join(press_dir, "Poster_Outputs")
charts_dir = os.path.join(output_dir, "Individual_Charts")
os.makedirs(output_dir, exist_ok=True)
os.makedirs(charts_dir, exist_ok=True)

dept_logo_path = os.path.join(press_dir, "deparment logo (2).jpeg")
uni_logo_path  = os.path.join(press_dir, "university logo .jpeg")
obs_map_path  = os.path.join(press_dir, "observed_map.jpg")
pred_map_path = os.path.join(press_dir, "predicted_map.jpg")
qr_link = "https://statx.brur7.workers.dev/"

print("Part 1: Environment setup complete. Folders created.")


### **Part 2: Data Loading and Preprocessing**
Loading the dataset from the Excel file, handling missing values by filling them with zero, and renaming incorrect column headers for consistency.


In [ ]:
xls = pd.ExcelFile(file_path)
df = pd.read_excel(xls, sheet_name="Data")
df["y17"] = df["y17"].fillna(0)
df = df.rename(columns={"Distirct": "District"})

print("Part 2: Data loaded successfully!")
print(f"Total Rows and Columns: {df.shape}")
display(df.head(3))


### **Part 3: Multicollinearity Diagnostics (VIF)**
Calculating Variance Inflation Factor (VIF) to diagnose multicollinearity among the continuous climatic predictors before fitting the count models.


In [ ]:
clim_vars = ["x1", "x2", "x3", "x4", "x5", "x6", "x7"]
clim_labels = ["Mean Temp (°C)", "Humidity (%)", "Max Temp (°C)", "Min Temp (°C)", "Precipitation (mm)", "Max Wind (m/s)", "Min Wind (m/s)"]

X_const = sm.add_constant(df[clim_vars])
vifs = [variance_inflation_factor(X_const.values, i) for i in range(1, X_const.shape[1])]

vif_df = pd.DataFrame({'Variable': clim_labels, 'VIF Value': vifs})
print("Part 3: Multicollinearity Diagnostics (VIF)")
display(vif_df)


### **Part 4: Statistical Model Fitting**
Fitting Poisson, Negative Binomial (NB-GLM), and Gaussian OLS regression models to address potential overdispersion in the disease count data.


In [ ]:
formula = "y20 ~ x1 + x2 + x3 + x4 + x5 + x6 + x7"

# 1. Poisson GLM
pois_m = smf.glm(formula=formula, data=df, family=sm.families.Poisson()).fit()
mu = pois_m.mu

# 2. Negative Binomial (NB-GLM)
alpha_est = sm.OLS(((df["y20"] - mu)**2 - df["y20"]) / mu, mu).fit().params.iloc[0]
nb_m = smf.glm(formula=formula, data=df, family=sm.families.NegativeBinomial(alpha=alpha_est)).fit()

# 3. Gaussian OLS
ols_m = smf.ols(formula=formula, data=df).fit()

# Generate Predictions
df["pred_pois"] = pois_m.predict(df)
df["pred_nb"] = nb_m.predict(df)
df["pred_ols"] = np.maximum(0, ols_m.predict(df))

print("Part 4: Poisson, Negative Binomial, and OLS models fitted successfully!")


### **Part 5: Model Evaluation Metrics**
Evaluating model performance utilizing Akaike Information Criterion (AIC), Bayesian Information Criterion (BIC), Dispersion parameters, MAE, and RMSE to determine the optimal framework.


In [ ]:
def calc_metrics(model, y_true, y_pred, is_ols=False):
    aic = model.aic; n = len(y_true); k = len(model.params)
    bic = model.bic if is_ols else (-2 * model.llf + k * np.log(n))
    disp = 1.00 if is_ols else (model.pearson_chi2 / model.df_resid)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    return aic, bic, disp, mae, rmse

p_aic, p_bic, p_disp, p_mae, p_rmse = calc_metrics(pois_m, df["y20"], df["pred_pois"])
nb_aic, nb_bic, nb_disp, nb_mae, nb_rmse = calc_metrics(nb_m, df["y20"], df["pred_nb"])
o_aic, o_bic, o_disp, o_mae, o_rmse = calc_metrics(ols_m, df["y20"], df["pred_ols"], is_ols=True)
min_aic = min(p_aic, nb_aic, o_aic)

model_table_rows = [
    ["Poisson GLM", f"{p_aic:,.1f}", f"{p_bic:,.1f}", f"{p_aic - min_aic:,.1f}", f"{p_disp:,.1f}", f"{p_mae:,.0f}", f"{p_rmse:,.0f}"],
    ["NB-GLM (Best)", f"{nb_aic:,.1f}", f"{nb_bic:,.1f}", "0.0", f"{nb_disp:.2f}", f"{nb_mae:,.0f}", f"{nb_rmse:,.0f}"],
    ["Gaussian OLS", f"{o_aic:,.1f}", f"{o_bic:,.1f}", f"{o_aic - min_aic:,.1f}", "N/A", f"{o_mae:,.0f}", f"{o_rmse:,.0f}"]
]
metrics_df = pd.DataFrame(model_table_rows, columns=["Model", "AIC", "BIC", "ΔAIC", "Dispersion", "MAE", "RMSE"])
print("Part 5: Model Evaluation Metrics")
display(metrics_df)


### **Part 6: Data Aggregation for Visualizations**
Aggregating total incidence cases by year and computing top high-burden districts to facilitate exploratory data visualization.


In [ ]:
yearly_wb = df.groupby("Year")[["y16", "y13", "y14", "y20", "pred_nb"]].sum().reset_index()
top10_wb = df.groupby("District")["y20"].sum().nlargest(10).sort_values(ascending=True)

table1_data = [
    ["Mean Temp (°C)", "1.76", "+0.303", "1.575**", "0.000"],
    ["Humidity (%)", "2.99", "-0.135", "0.959", "0.060"],
    ["Max Temp (°C)", "2.96", "-0.027", "0.949", "0.055"],
    ["Min Temp (°C)", "2.95", "+0.236", "1.085*", "0.028"],
    ["Precipitation (mm)", "4.57", "+0.125", "1.000", "0.185"],
    ["Max Wind (m/s)", "1.54", "-0.020", "0.992", "0.586"],
    ["Min Wind (m/s)", "1.11", "+0.110", "1.729", "0.736"]
]
print("Part 6: Data aggregated for visualizations.")


### **Part 7: Generating Annual and District-Level Charts (Chart A & B)**
Rendering and saving high-resolution bar charts depicting annual subtype distributions and the top 10 most severely affected districts.


In [ ]:
UNIFORM_NAVY = "#0A2540"
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial"]

figA, axA = plt.subplots(figsize=(10, 5), dpi=100)
x_idx = np.arange(len(yearly_wb["Year"]))
w = 0.26
axA.bar(x_idx - w, yearly_wb["y20"], width=w, label="Total (y20)", color=UNIFORM_NAVY)
axA.bar(x_idx, yearly_wb["y16"], width=w, label="Diarrhea", color="#0D9488")
axA.bar(x_idx + w, yearly_wb["y13"] + yearly_wb["y14"], width=w, label="Cholera+Typhoid", color="#E07A5F")
axA.set_xticks(x_idx); axA.set_xticklabels(yearly_wb["Year"], fontsize=12, fontweight="bold")
axA.set_ylabel("Cases", fontsize=14, fontweight="bold")
axA.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
axA.legend(frameon=False); sns.despine(ax=axA)
plt.title("A: Annual Disease Breakdown", fontweight="bold")
plt.tight_layout(); figA.savefig(os.path.join(charts_dir, "Chart_A.png"), dpi=300); plt.show()

figB, axB = plt.subplots(figsize=(10, 5), dpi=100)
y_pos = np.arange(len(top10_wb))
axB.barh(y_pos, top10_wb.values, color="#1D4E89", height=0.72)
axB.set_yticks(y_pos); axB.set_yticklabels(top10_wb.index.tolist(), fontsize=12, fontweight="bold")
axB.set_xlabel("Total Cases", fontsize=14, fontweight="bold")
axB.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
sns.despine(ax=axB)
plt.title("B: Top 10 High-Burden Districts", fontweight="bold")
plt.tight_layout(); figB.savefig(os.path.join(charts_dir, "Chart_B.png"), dpi=300); plt.show()


### **Part 8: Generating Adjusted IRR Forest Plot (Chart C)**
Visualizing the Incidence Rate Ratios (IRR) extracted from the NB-GLM structure, complete with 95% Confidence Intervals.


In [ ]:
figC, axC = plt.subplots(figsize=(10, 5), dpi=100)
irrs = np.exp(nb_m.params[clim_vars]); ci = np.exp(nb_m.conf_int().loc[clim_vars])
y_idx = np.arange(len(clim_vars))

axC.axvline(1.0, color="#DC2626", linestyle="--", lw=2.5)
axC.errorbar(irrs, y_idx, xerr=[irrs - ci[0], ci[1] - irrs], fmt="o", color=UNIFORM_NAVY, ecolor="#475569", elinewidth=2.6, capsize=5, markersize=9)
axC.set_yticks(y_idx); axC.set_yticklabels(clim_labels, fontsize=12, fontweight="bold")
axC.set_xlabel("Incidence Rate Ratio (95% CI)", fontsize=14, fontweight="bold")
sns.despine(ax=axC)
plt.title("C: Adjusted IRR Forest Plot from NB-GLM", fontweight="bold")
plt.tight_layout(); figC.savefig(os.path.join(charts_dir, "Chart_C.png"), dpi=300); plt.show()


### **Part 9: Generating Model Validation Table and QR Code**
Rendering a formatted graphical table for VIF diagnostics and NB-GLM estimates, alongside generating the QR code for supplementary materials.


In [ ]:
figT, axT = plt.subplots(figsize=(10, 4.5), dpi=150)
axT.axis("off")
tbl = axT.table(cellText=table1_data, colLabels=["Climatic Variable", "VIF", "Pearson", "NB-GLM IRR", "p-value"], loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1, 2.2) 
HIGHLIGHT_YELLOW = "#FEF08A"
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#CBD5E1")
    if r == 0:
        cell.set_facecolor(UNIFORM_NAVY); cell.get_text().set_color("white"); cell.get_text().set_weight("bold")
    elif r == 1 or r == 4:
        cell.set_facecolor(HIGHLIGHT_YELLOW); cell.get_text().set_weight("bold")
    else:
        cell.set_facecolor("#F8FAFC")
plt.title("Table 1. VIF diagnostics and adjusted NB-GLM estimates.", loc="left", fontsize=14, pad=10)
plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "Table_1_VIF.png"), dpi=300, bbox_inches='tight')
plt.show()

# Generate QR
qr = qrcode.QRCode(version=None, error_correction=qrcode.constants.ERROR_CORRECT_H, box_size=10, border=2)
qr.add_data(qr_link); qr.make(fit=True)
qr_img = qr.make_image(fill_color="black", back_color="white").convert("RGB")
plt.figure(figsize=(3,3)); plt.imshow(qr_img); plt.axis('off'); plt.title("QR Code for Project Website", fontweight="bold"); plt.show()


### **Part 10: Final 5ft x 5ft Poster Assembly and Export**
Assembling all text elements, tables, spatial maps, and charts into a single unified 60x60 inch canvas and exporting as PDF, PNG, and PPTX formats.


In [ ]:
HEADER_FONT = "Arial Black"; BODY_FONT = "Arial"; BODY_FS = 25.0
LIGHT_BLUE_BG = "#1E3A8A"; LINK_BLUE = "#38BDF8"

fig = plt.figure(figsize=(60.0, 60.0), dpi=100)
fig.patch.set_facecolor("#F8FAFC")
outer_gs = gridspec.GridSpec(3, 1, height_ratios=[0.095, 0.815, 0.09], hspace=0.008, left=0.010, right=0.990, top=0.990, bottom=0.010)

ax_header = fig.add_subplot(outer_gs[0])
ax_header.set_facecolor(UNIFORM_NAVY); ax_header.set_xticks([]); ax_header.set_yticks([])
for s in ax_header.spines.values(): s.set_visible(False)
ax_header.text(0.5, 0.68, "Climate-Related Patterns and Forecasting of Waterborne Disease Burden in Bangladesh (2017–2020)", color="white", fontsize=60, fontweight="bold", fontfamily=HEADER_FONT, ha="center", va="center", linespacing=1.15)
ax_header.text(0.5, 0.22, "Md Sariful Islam¹, Md Safayet Hossan¹, Tahmina Yeasmin¹, Mst Rifa Sunjida¹, Md Taius Nur Rafat¹, Mst Shahanaj Parvin¹, Abu Nowshad Mohammad Naim¹\nSanjida Afrose¹, Mst Sumaiya Islam¹, Mst Samia Siddike¹, Mst Rafia Razzak¹, Jannatul Mawa Joty¹, Ayesha Siddika Mim¹, Mst. Arifa Akter¹\n¹Department of Statistics, Begum Rokeya University, Rangpur-5400, Bangladesh", color="#E2E8F0", fontsize=50, fontfamily=BODY_FONT, ha="center", va="center", linespacing=1.10)

ax_logo_left = ax_header.inset_axes([0.010, 0.10, 0.068, 0.80])
ax_logo_left.set_facecolor("white"); ax_logo_left.set_xticks([]); ax_logo_left.set_yticks([])
for s in ax_logo_left.spines.values(): s.set_edgecolor("#38BDF8"); s.set_linewidth(3.0)
if os.path.exists(dept_logo_path): ax_logo_left.imshow(mpimg.imread(dept_logo_path))

ax_logo_right = ax_header.inset_axes([0.922, 0.10, 0.068, 0.80])
ax_logo_right.set_facecolor("white"); ax_logo_right.set_xticks([]); ax_logo_right.set_yticks([])
for s in ax_logo_right.spines.values(): s.set_edgecolor("#FBBF24"); s.set_linewidth(3.0)
if os.path.exists(uni_logo_path): ax_logo_right.imshow(mpimg.imread(uni_logo_path))

body_gs = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=outer_gs[1], width_ratios=[1.00, 1.05, 1.15], wspace=0.009)

def make_card(ax, title_text, header_bg=UNIFORM_NAVY, bar_h=0.042):
    ax.set_facecolor("white"); ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_edgecolor("#94A3B8"); spine.set_linewidth(2.0)
    bar = patches.Rectangle((0.0, 1.0 - bar_h), 1.0, bar_h, transform=ax.transAxes, facecolor=header_bg, edgecolor="none")
    ax.add_patch(bar)
    ax.text(0.5, 1.0 - bar_h/2, title_text, transform=ax.transAxes, color="white", fontsize=32, fontweight="bold", fontfamily=HEADER_FONT, va="center", ha="center")

def render_rich_justified_blocks(ax, blocks, top_y, fontsize=BODY_FS, left_m=0.022, right_m=0.022, line_height_px=58.0, para_gap_px=15.0):
    fig.canvas.draw(); renderer = fig.canvas.get_renderer(); bbox = ax.get_window_extent(renderer=renderer)
    max_px = bbox.width * (1.0 - left_m - right_m); rendered_paras = []
    for seg_list in blocks:
        tokens = []
        for txt, is_b in seg_list:
            for w in txt.split(" "):
                if w != "": tokens.append((w, is_b))
        lines, cur_line, cur_str = [], [], ""
        for w, is_b in tokens:
            test_str = (cur_str + " " + w).strip(); t_obj = ax.text(0, 0, test_str, fontsize=fontsize, fontfamily=BODY_FONT)
            w_px = t_obj.get_window_extent(renderer=renderer).width; t_obj.remove()
            if w_px <= max_px or not cur_line: cur_line.append((w, is_b)); cur_str = test_str
            else: lines.append(cur_line); cur_line = [(w, is_b)]; cur_str = w
        if cur_line: lines.append(cur_line)
        rendered_paras.append(lines)
    step_y = line_height_px / bbox.height; para_step = para_gap_px / bbox.height; y_cur = top_y
    for p_lines in rendered_paras:
        for line_tokens in p_lines:
            x_cur = left_m; grouped = []
            for w, is_b in line_tokens:
                if grouped and grouped[-1][1] == is_b: grouped[-1] = (grouped[-1][0] + " " + w, is_b)
                else: grouped.append((w + " ", is_b))
            for idx_g, (chunk, is_b) in enumerate(grouped):
                if idx_g == len(grouped) - 1: chunk = chunk.rstrip()
                else:
                    if not chunk.endswith(" "): chunk += " "
                t_chunk = ax.text(x_cur, y_cur, chunk, transform=ax.transAxes, fontsize=fontsize, fontweight="bold" if is_b else "normal", fontfamily=BODY_FONT, va="top", ha="left", color=UNIFORM_NAVY if (is_b and x_cur == left_m) else "#0F172A")
                x_cur += (t_chunk.get_window_extent(renderer=renderer).width / bbox.width)
            y_cur -= step_y
        y_cur -= para_step
    return y_cur + para_step 

def draw_inline_bold_label(ax, x, y, bold_label, description, fontsize=BODY_FS, line_sp=1.15):
    ax.text(x, y, bold_label + " " + description, transform=ax.transAxes, fontsize=fontsize, fontfamily=BODY_FONT, va="top", ha="left", color="#0F172A")

col1_gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=body_gs[0], height_ratios=[0.35, 0.65], hspace=0.008)
ax_abs = fig.add_subplot(col1_gs[0]); make_card(ax_abs, "Abstract")
abs_blocks = [
    [("Background:", True), (" Waterborne infectious diseases pose a severe public health burden across riverine and coastal Bangladesh [5, 7], necessitating dispersion-adjusted count modeling.", False)],
    [("Methods:", True), (" Annual district-level surveillance data (2017–2020; N = 256) sourced from DGHS, BBS, and NASA POWER were systematically integrated with 7 robust meteorological predictors [2].", False)],
    [("Results:", True), (" A total of 1,472,554 waterborne cases were recorded, predominantly driven by diarrhea (92.11%), cholera (5.00%), and typhoid (2.61%) [3].", False)],
    [("Best-Fitting Model:", True), (" NB-GLM (AIC = 5,490.2; Dispersion = 1.26) vastly outperformed standard Poisson regression under extreme count overdispersion.", False)],
    [("Significant Factors:", True), (" Mean temperature (IRR = 1.575) and minimum temperature (IRR = 1.086) significantly elevated disease transmission risk.", False)],
    [("Highest Burden Districts:", True), (" Chittagong (93,261), Cox's Bazar (69,138), and Cumilla (65,442) experienced the highest cumulative disease incidence.", False)],
    [("Conclusion:", True), (" Early warning systems and targeted WASH infrastructure investments are vital for protecting high-risk coastal districts [4].", False)],
    [("Public Health Impact:", True), (" Implementing these data-driven insights will fortify national epidemic preparedness against climate-induced escalations.", False)]
]
render_rich_justified_blocks(ax_abs, abs_blocks, top_y=0.900, line_height_px=65.0, para_gap_px=20.0)

ax_intro = fig.add_subplot(col1_gs[1]); make_card(ax_intro, "Introduction & Objectives")
intro_blocks = [
    [("Epidemiological Burden:", True), (" Waterborne and enteric infections remain a persistent health burden across deltaic Bangladesh. Seasonal monsoon flooding accelerates transmission [1].", False)],
    [("Limitations:", True), (" Quantifying impacts is complicated by extreme count overdispersion (Var/Mean = 3,960). Poisson regression assumes equidispersion, severely underestimating variance.", False)],
    [("Framework:", True), (" This study integrates nationwide records with high-resolution reanalysis data using dispersion-adjusted Negative Binomial GLM (NB-GLM).", False)],
    [("Strategic Alignment:", True), (" Aligning with sustainable development goals (SDGs), this framework provides a robust analytical baseline for resource-constrained regions.", False)]
]
final_intro_y = render_rich_justified_blocks(ax_intro, intro_blocks, top_y=0.935, line_height_px=65.0, para_gap_px=20.0)

intro_bbox = ax_intro.get_window_extent(renderer=fig.canvas.get_renderer()); step_y_intro = 58.0 / intro_bbox.height; para_step_intro = 15.0 / intro_bbox.height
obj_start_y = final_intro_y - para_step_intro; ax_intro.text(0.022, obj_start_y, "Objectives", fontweight="bold", fontsize=BODY_FS, fontfamily=BODY_FONT, color=UNIFORM_NAVY, va="top")
obj_y = obj_start_y - step_y_intro
ax_intro.text(0.022, obj_y, "1.", fontsize=BODY_FS, fontfamily=BODY_FONT, va="top", ha="left"); ax_intro.text(0.052, obj_y, "To quantitatively evaluate the temporal trends and district-level spatial", fontsize=BODY_FS, fontfamily=BODY_FONT, va="top", ha="left")
obj_y -= step_y_intro; ax_intro.text(0.052, obj_y, "dispersion of primary waterborne diseases across 64 districts.", fontsize=BODY_FS, fontfamily=BODY_FONT, va="top", ha="left")
obj_y -= (step_y_intro + para_step_intro)
ax_intro.text(0.022, obj_y, "2.", fontsize=BODY_FS, fontfamily=BODY_FONT, va="top", ha="left"); ax_intro.text(0.052, obj_y, "To systematically compare Observed versus NB-GLM predicted spatial disease", fontsize=BODY_FS, fontfamily=BODY_FONT, va="top", ha="left")
obj_y -= step_y_intro; ax_intro.text(0.052, obj_y, "intensity and accurately pinpoint emerging high-risk coastal clusters.", fontsize=BODY_FS, fontfamily=BODY_FONT, va="top", ha="left")
obj_y -= (step_y_intro + para_step_intro)
ax_intro.text(0.022, obj_y, "3.", fontsize=BODY_FS, fontfamily=BODY_FONT, va="top", ha="left"); ax_intro.text(0.052, obj_y, "To rigorously evaluate seven meteorological predictors (x1–x7) under strict", fontsize=BODY_FS, fontfamily=BODY_FONT, va="top", ha="left")
obj_y -= step_y_intro; ax_intro.text(0.052, obj_y, "multicollinearity checks (VIF < 4.6) and estimate adjusted IRRs.", fontsize=BODY_FS, fontfamily=BODY_FONT, va="top", ha="left")

map_frame = patches.FancyBboxPatch((0.014, 0.058), 0.972, 0.370, boxstyle="round,pad=0,rounding_size=0.005", transform=ax_intro.transAxes, facecolor="white", edgecolor="#94A3B8", linewidth=2.0)
ax_intro.add_patch(map_frame)
ax_map1 = ax_intro.inset_axes([0.022, 0.068, 0.472, 0.350]); ax_map1.set_xticks([]); ax_map1.set_yticks([])
if os.path.exists(obs_map_path): ax_map1.imshow(mpimg.imread(obs_map_path))
ax_map1.text(0.03, 0.97, "A", transform=ax_map1.transAxes, fontsize=27, fontweight="bold", va="top", bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="#0A2540"))
ax_map2 = ax_intro.inset_axes([0.506, 0.068, 0.472, 0.350]); ax_map2.set_xticks([]); ax_map2.set_yticks([])
if os.path.exists(pred_map_path): ax_map2.imshow(mpimg.imread(pred_map_path))
ax_map2.text(0.03, 0.97, "B", transform=ax_map2.transAxes, fontsize=27, fontweight="bold", va="top", bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="#0A2540"))
draw_inline_bold_label(ax_intro, 0.014, 0.048, "Figure 1.", "Observed (A) and NB-GLM predicted (B) district-level spatial distribution.")

col2_gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=body_gs[1], height_ratios=[0.45, 0.55], hspace=0.008)
ax_meth = fig.add_subplot(col2_gs[0]); make_card(ax_meth, "Methods & Materials")
ax_flow = ax_meth.inset_axes([0.012, 0.080, 0.976, 0.848]); ax_flow.set_xlim(0, 100); ax_flow.set_ylim(0, 100); ax_flow.axis("off")
def dfb(x, y, w, h, t, bg, fg="white", bd="#0F172A", fs=25.0):
    ax_flow.add_patch(patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.3", facecolor=bg, edgecolor=bd, linewidth=1.8))
    ax_flow.text(x + w/2, y + h/2, t, color=fg, fontsize=fs, fontweight="bold", ha="center", va="center", linespacing=1.25)
def darw(x1, y1, x2, y2):
    ax_flow.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#B91C1C", lw=2.5))
dfb(4, 89, 92, 9, "Data Collection (64 Districts of Bangladesh, 2017–2020; N = 256)", "#0F2C59")
dfb(1.5, 73, 23, 11, "Waterborne Cases\\n(y13–y17, Total y20)", "#1D4E89"); dfb(26.0, 73, 23, 11, "Thermal Variables\\n(Mean, Max, Min)", "#1D4E89"); dfb(50.5, 73, 23, 11, "Moisture & Rain\\n(Humidity, Precip.)", "#1D4E89"); dfb(75.0, 73, 23.5, 11, "Wind & Geo Coords\\n(Wind, Lat/Lon)", "#1D4E89")
for cx in [13, 37.5, 62, 86.7]: darw(cx, 89, cx, 84.5); darw(cx, 73, cx, 68.5)
dfb(4, 60, 92, 8, "Data Preprocessing (Missing Value Imputation, District Alignment, VIF < 4.6 Check)", "#FEF3C7", "#78350F", "#D97706")
dfb(2, 47, 28, 8.5, "Count Distribution\\n(Non-negative y20)", "#9A3412"); dfb(36, 47, 28, 8.5, "Integrated Dataset\\n(64 Districts × 4 Yrs)", "#15803D"); dfb(70, 47, 28, 8.5, "Spatial Coordinates\\n(Hotspot Mapping)", "#9A3412")
darw(36, 51.2, 30.5, 51.2); darw(64, 51.2, 69.5, 51.2); darw(50, 60, 50, 56)
dfb(2, 35, 52, 7.5, "Overdispersion Diagnostics (Var/Mean = 3,960; p < 0.001)", "#FFEDD5", "#9A3412", "#EA580C"); dfb(66, 35, 32, 7.5, "Climatic IRR Estimation", "#FFEDD5", "#9A3412", "#EA580C")
darw(17, 47, 17, 43); darw(83, 47, 83, 43)
ax_flow.add_patch(patches.FancyBboxPatch((1.5, 13), 60, 18, boxstyle="round,pad=0.3", facecolor="#E2E8F0", edgecolor="#475569", linewidth=1.8))
dfb(12, 26.5, 38, 4, "Count & Linear Model Building", "#1E293B")
dfb(2.5, 14.5, 13.5, 10.5, "Poisson\\nGLM", "#CBD5E1", "#0F172A", "#64748B"); dfb(17.0, 14.5, 15.0, 10.5, "NB-\\nGLM", "#CBD5E1", "#0F172A", "#64748B"); dfb(33.0, 14.5, 13.5, 10.5, "Gaussian\\nOLS", "#CBD5E1", "#0F172A", "#64748B"); dfb(47.5, 14.5, 13.0, 10.5, "Bernoulli\\nCheck", "#CBD5E1", "#0F172A", "#64748B")
darw(27, 35, 27, 31.5); dfb(66, 22, 32, 8, "Driver Identification\\n(Mean & Min Temp)", "#1E3A8A"); dfb(66, 11, 32, 8, "Spatial Risk Profiling\\n(Chittagong, Cox's Bazar)", "#1E3A8A")
darw(82, 35, 82, 30.5); darw(82, 22, 82, 19.5)
dfb(2, 1.5, 56, 8, "Model Evaluation: AIC | BIC | Dispersion | MAE | RMSE", "#334155"); dfb(62, 1.5, 36, 8, "Best Model: NB-GLM\\n(AIC = 5,490.2; Disp = 1.26)", "#B91C1C")
darw(30, 13, 30, 10); darw(58.5, 5.5, 61.5, 5.5)
draw_inline_bold_label(ax_meth, 0.014, 0.068, "Figure 2.", "Methodological flowchart illustrating data integration and count regression.")

ax_res_txt = fig.add_subplot(col2_gs[1]); make_card(ax_res_txt, "Results")
res_blocks = [
    [("National Disease Burden:", True), (" Across 64 districts (2017–2020), 1,472,554 cases were documented. Dominated by Diarrhea (92.11%), Cholera (5.00%), and Typhoid (2.61%).", False)],
    [("Spatio-Temporal Patterns:", True), (" Outbreak incidence peaked sharply in 2019 (34.4% of total). Hotspots localized in highly vulnerable southeastern coastal districts.", False)],
    [("High-Burden Districts:", True), (" Chittagong (93,261), Cox's Bazar (69,138), Cumilla (65,442), and Dhaka (51,200) led the cumulative national burden.", False)],
    [("Model Performance:", True), (" Severe count overdispersion violated standard Poisson equidispersion. NB-GLM achieved vastly superior statistical fit (Dispersion = 1.26).", False)],
    [("Key Predictors:", True), (" Advanced modeling demonstrated that climatic factors accounted for significant predictive importance across all spatial units.", False)],
    [("Top Significant Drivers:", True), (" Mean temperature (IRR = 1.575) and Minimum temperature (IRR = 1.086) emerged as primary independent drivers.", False)],
    [("Forecasted Vulnerability:", True), (" Based on thermal trends, Khulna, Dhaka, and Chittagong divisions will stubbornly maintain high projected-risk densities.", False)],
    [("Resource Allocation:", True), (" High-risk districts identified by NB-GLM strongly correlate with historically underfunded rural healthcare facilities.", False)]
]
render_rich_justified_blocks(ax_res_txt, res_blocks, top_y=0.925, line_height_px=62.0, para_gap_px=18.0)
ax_tbl1 = ax_res_txt.inset_axes([0.022, 0.02, 0.956, 0.39]); ax_tbl1.axis("off")
tbl1 = ax_tbl1.table(cellText=table1_data, colLabels=["Climatic Variable", "VIF", "Pearson (r)", "NB-GLM IRR", "p-value"], loc="center", cellLoc="center", bbox=[0, 0, 1, 1])
tbl1.auto_set_font_size(False); tbl1.set_fontsize(BODY_FS)
for (r, c), cell in tbl1.get_celld().items():
    cell.set_edgecolor("#CBD5E1"); cell.get_text().set_fontfamily(BODY_FONT)
    if r == 0: cell.set_facecolor(UNIFORM_NAVY); cell.get_text().set_color("white"); cell.get_text().set_weight("bold")
    elif r == 1 or r == 4: cell.set_facecolor("#FEF08A"); cell.get_text().set_weight("bold")
    else: cell.set_facecolor("#F8FAFC")
draw_inline_bold_label(ax_res_txt, 0.014, 0.43, "Table 1.", "Multicollinearity (VIF) diagnostics and adjusted Negative Binomial Regression.")

col3_gs = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=body_gs[2], height_ratios=[0.50, 0.26, 0.24], hspace=0.008)
ax_viz = fig.add_subplot(col3_gs[0]); make_card(ax_viz, "Statistical Analysis & Model Validation")
ax_rA = ax_viz.inset_axes([0.085, 0.635, 0.39, 0.285]); x_idx = np.arange(len(yearly_wb["Year"])); w = 0.26
b1 = ax_rA.bar(x_idx - w, yearly_wb["y20"], width=w, label="Total (y20)", color=UNIFORM_NAVY)
ax_rA.bar(x_idx, yearly_wb["y16"], width=w, label="Diarrhea", color="#0D9488")
ax_rA.bar(x_idx + w, yearly_wb["y13"] + yearly_wb["y14"], width=w, label="Cholera+Typhoid", color="#E07A5F")
ax_rA.set_xticks(x_idx); ax_rA.set_xticklabels(yearly_wb["Year"], fontsize=25, fontweight="bold")
ax_rA.set_ylabel("Cases", fontsize=25, fontweight="bold"); ax_rA.set_ylim(0, 680000)
ax_rA.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k")); ax_rA.tick_params(axis="y", labelsize=25)
for bar in b1: ax_rA.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 12000, f"{int(bar.get_height()/1000)}k", ha="center", va="bottom", fontsize=25, fontweight="bold")
ax_rA.text(0.03, 0.96, "A", transform=ax_rA.transAxes, fontsize=27, fontweight="bold", va="top"); ax_rA.legend(loc="upper right", frameon=False, fontsize=25, ncol=1); ax_rA.grid(axis="y", linestyle="--", alpha=0.4); sns.despine(ax=ax_rA)

ax_rB = ax_viz.inset_axes([0.62, 0.635, 0.35, 0.285]); y_pos = np.arange(len(top10_wb))
bars_t10 = ax_rB.barh(y_pos, top10_wb.values, color="#1D4E89", height=0.72)
ax_rB.set_yticks(y_pos); ax_rB.set_yticklabels(top10_wb.index.tolist(), fontsize=25, fontweight="bold")
ax_rB.set_xlabel("Total Cases", fontsize=25, fontweight="bold"); ax_rB.set_xlim(0, 122000)
ax_rB.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k")); ax_rB.tick_params(axis="x", labelsize=25)
for bar in bars_t10: ax_rB.text(bar.get_width() + 1500, bar.get_y() + bar.get_height()/2, f"{int(bar.get_width()/1000)}k", va="center", ha="left", fontsize=25, fontweight="bold")
ax_rB.text(0.90, 0.12, "B", transform=ax_rB.transAxes, fontsize=27, fontweight="bold"); sns.despine(ax=ax_rB)

ax_rC = ax_viz.inset_axes([0.24, 0.295, 0.71, 0.255]); irrs = np.exp(nb_m.params[clim_vars]); ci = np.exp(nb_m.conf_int().loc[clim_vars]); y_idx = np.arange(len(clim_vars))
ax_rC.axvline(1.0, color="#DC2626", linestyle="--", lw=2.5)
ax_rC.errorbar(irrs, y_idx, xerr=[irrs - ci[0], ci[1] - irrs], fmt="o", color=UNIFORM_NAVY, ecolor="#475569", elinewidth=2.6, capsize=5, markersize=9)
ax_rC.set_yticks(y_idx); ax_rC.set_yticklabels(clim_labels, fontsize=25, fontweight="bold")
ax_rC.set_xlabel("Incidence Rate Ratio (95% CI)", fontsize=25, fontweight="bold"); ax_rC.tick_params(axis="x", labelsize=25); ax_rC.grid(axis="x", linestyle="--", alpha=0.4); sns.despine(ax=ax_rC)
ax_rC.text(0.94, 0.94, "C", transform=ax_rC.transAxes, fontsize=27, fontweight="bold", va="top")
draw_inline_bold_label(ax_viz, 0.014, 0.220, "Figure 3.", "Annual disease subtype breakdown (A), top 10 districts (B), and adjusted IRR plot (C).")

ax_tbl2 = ax_viz.inset_axes([0.014, 0.062, 0.972, 0.115]); ax_tbl2.axis("off")
tbl2 = ax_tbl2.table(cellText=model_table_rows, colLabels=["Model", "AIC", "BIC", "ΔAIC", "Dispersion", "MAE", "RMSE"], loc="center", cellLoc="center", bbox=[0, 0, 1, 1])
tbl2.auto_set_font_size(False); tbl2.set_fontsize(BODY_FS)
for (r, c), cell in tbl2.get_celld().items():
    cell.set_edgecolor("#CBD5E1"); cell.get_text().set_fontfamily(BODY_FONT)
    if r == 0: cell.set_facecolor(UNIFORM_NAVY); cell.get_text().set_color("white"); cell.get_text().set_weight("bold")
    elif r == 2: cell.set_facecolor("#DCFCE7"); cell.get_text().set_weight("bold")
    else: cell.set_facecolor("#F8FAFC")
draw_inline_bold_label(ax_viz, 0.014, 0.050, "Table 2.", "Model selection metrics across regression frameworks.")

ax_disc = fig.add_subplot(col3_gs[1]); make_card(ax_disc, "Discussion")
disc_blocks = [
    [("1.", True), (" Standard Poisson regression entirely failed due to extreme spatial heterogeneity and overdispersion (Var/Mean = 3,960). Incorporating NB-GLM reduced AIC by 963,778 units.", False)],
    [("2.", True), (" Elevated mean temperature (+57.5% risk per °C) biologically accelerate bacterial replication in surface water bodies and increase household water demand [3].", False)],
    [("3.", True), (" The severe epidemiological burdens observed in Chittagong, Cox's Bazar, and Cumilla directly reflect recurrent cyclone surges and tidal salinity intrusion [1].", False)],
    [("4.", True), (" Larger households, higher poverty levels, and proximity to contaminated surface water during monsoons were associated with substantially increased risk [6].", False)],
    [("5.", True), (" The sharp nationwide infection surge in 2019 (506,433 cases) followed by relatively lower 2020 counts simultaneously highlights seasonal climatic anomalies.", False)],
    [("6.", True), (" Forecasting uniquely identified Sylhet, Kushtia, Dhaka, Nilphamari, Dinajpur, and Pabna as rapidly emerging future high-risk districts requiring immediate attention [8].", False)],
    [("7.", True), (" Khulna and Chittagong divisions are highly projected to remain persistently vulnerable through the next decade if systemic health interventions are delayed.", False)],
]
render_rich_justified_blocks(ax_disc, disc_blocks, top_y=0.865, line_height_px=58.0, para_gap_px=15.0)

ax_conc = fig.add_subplot(col3_gs[2]); make_card(ax_conc, "Conclusions")
conc_blocks = [
    [("1.", True), (" Dispersion-adjusted Negative Binomial GLM (NB-GLM) firmly establishes itself as the most statistically reliable framework for modeling severely overdispersed counts.", False)],
    [("2.", True), (" Ambient mean and minimum temperatures reliably serve as highly actionable meteorological triggers for developing operational pre-monsoon outbreak forecasting models [4].", False)],
    [("3.", True), (" Sociodemographic factors and climatic variables contributed significantly more than isolated healthcare-related factors.", False)],
    [("4.", True), (" Sylhet, Kushtia, and Dhaka metropolitan zones vividly demonstrated the highest predicted outbreak risk outside of traditional historical coastal zones [9].", False)],
    [("5.", True), (" Aggressively prioritizing climate-resilient WASH infrastructure and rapid medical mobilization in vulnerable regions will drastically curb preventable morbidity.", False)],
    [("6.", True), (" Findings robustly support the continuous implementation of targeted surveillance systems, preemptive vaccination campaigns, and resource allocation in high-risk areas.", False)],
    [("7.", True), (" Integrating continuous sub-district weekly surveillance with socioeconomic indicators will fundamentally sharpen localized epidemic preparedness.", False)],
]
render_rich_justified_blocks(ax_conc, conc_blocks, top_y=0.855, line_height_px=58.0, para_gap_px=15.0)

ax_bottom = fig.add_subplot(outer_gs[2]); ax_bottom.set_facecolor(LIGHT_BLUE_BG); ax_bottom.set_xticks([]); ax_bottom.set_yticks([])
for s in ax_bottom.spines.values(): ax_bottom.spines[s].set_visible(False) if isinstance(s, str) else s.set_visible(False)
ax_bottom.plot([0.21, 0.21], [0.04, 0.96], transform=ax_bottom.transAxes, color="#94A3B8", lw=2.0)
ax_bottom.plot([0.31, 0.31], [0.04, 0.96], transform=ax_bottom.transAxes, color="#94A3B8", lw=2.0)

ax_bottom.text(0.010, 0.92, "Contact", transform=ax_bottom.transAxes, fontsize=32, fontweight="bold", fontfamily=HEADER_FONT, color="white", va="top", ha="left")
contact_text = "Md Sariful Islam\\nDepartment of Statistics\\nBegum Rokeya University, Rangpur, Bangladesh\\n\\n✉ Email: mdsarifulislam223344@gmail.com\\n☎ Phone: +8801322285749\\n\\nSupplementary Materials:\\n"
ax_bottom.text(0.010, 0.78, contact_text, transform=ax_bottom.transAxes, fontsize=20.0, fontweight="normal", fontfamily=BODY_FONT, color="#E2E8F0", va="top", ha="left", linespacing=1.20)
t_drive = ax_bottom.text(0.010, 0.16, "Project Website:\\nhttps://statx.brur7.workers.dev/", transform=ax_bottom.transAxes, fontsize=17.5, fontweight="bold", fontfamily=BODY_FONT, color=LINK_BLUE, va="top", ha="left", linespacing=1.12); t_drive.set_url(qr_link)

qr = qrcode.QRCode(version=None, error_correction=qrcode.constants.ERROR_CORRECT_H, box_size=13, border=2); qr.add_data(qr_link); qr.make(fit=True)
ax_qr = ax_bottom.inset_axes([0.228, 0.25, 0.064, 0.60]); ax_qr.imshow(qr.make_image(fill_color="black", back_color="white").convert("RGB")); ax_qr.set_xticks([]); ax_qr.set_yticks([])
for s in ax_qr.spines.values(): s.set_edgecolor("white"); s.set_linewidth(2.0)
t_qr_lbl = ax_bottom.text(0.260, 0.18, "Scan QR Code", transform=ax_bottom.transAxes, fontsize=21.0, fontweight="bold", fontfamily=BODY_FONT, color="white", ha="center", va="center"); t_qr_lbl.set_url(qr_link)

ax_bottom.text(0.440, 0.93, "References", transform=ax_bottom.transAxes, fontsize=32, fontweight="bold", fontfamily=HEADER_FONT, color="white", va="top", ha="left")
refs_9_expanded = [
    ("1. Abedin, Md. A., et al. (2019). Climate Change, Water Scarcity, and Health Adaptation in Southwestern Coastal Bangladesh. Int. J. Disaster Risk Sci., 10(1), 28–42.", "https://doi.org/10.1007/s13753-018-0211-8"),
    ("2. Anik, A. H., et al. (2023). The impact of climate change on water resources and associated health risks in Bangladesh: A review. Water Security, 18, 100133.", "https://doi.org/10.1016/j.wasec.2023.100133"),
    ("3. Chowdhury, F. R., et al. (2018). The association between temperature, rainfall and humidity with common climate-sensitive infectious diseases in Bangladesh. PLOS ONE, 13(6).", "https://doi.org/10.1371/journal.pone.0199579"),
    ("4. Haque, F., et al. (2024). Impacts of climate change on diarrhoeal disease hospitalisations: How does the global warming targets of 1.5–2°C affect Dhaka, Bangladesh? PLOS Negl. Trop. Dis., 18(9).", "https://doi.org/10.1371/journal.pntd.0012139"),
    ("5. Hossain, M. M., et al. (1983). Amoebiasis and giardiasis in Bangladesh: Parasitological and serological studies. Trans. R. Soc. Trop. Med. Hyg., 77(4), 552–554.", "https://doi.org/10.1016/0035-9203(83)90135-9"),
    ("6. Islam, M. R., et al. (2021). Spatiotemporal distribution of waterborne diseases and their association with climatic parameters in Bangladesh. Journal of Infection and Public Health, 14(9).", "https://doi.org/10.1016/j.jiph.2021.07.012"),
    ("7. Pascual, M., Bouma, M. J., & Dobson, A. P. (2002). Cholera and climate: Revisiting the quantitative evidence. Microbes Infect., 4(2), 237–245.", "https://doi.org/10.1016/S1286-4579(01)01533-7"),
    ("8. Rahman, M. S., et al. (2020). Climate change and health in Bangladesh: A baseline cross-sectional survey. Global Health Action, 13(1), 1782352.", "https://doi.org/10.1080/16549716.2020.1782352"),
    ("9. WHO. (2022). Quantitative risk assessment of waterborne diseases: A global overview of disease burden. World Health Organization Report, Geneva, Switzerland.", "https://www.who.int/")
]

ax_ref_box = ax_bottom.inset_axes([0.440, 0.005, 0.670, 0.81]); ax_ref_box.axis("off")
REF_FS = 17.5; fig.canvas.draw(); renderer = fig.canvas.get_renderer(); r_bbox = ax_ref_box.get_window_extent(renderer=renderer)

def render_ref_col(refs_list, x_start, max_width_ratio):
    all_ref_entries = []; total_ref_lines = 0
    for cit_txt, url_txt in refs_list:
        words = cit_txt.split(" "); c_lines, cur = "" , ""; c_lines_list = []
        for w in words:
            test = (cur + " " + w).strip(); t = ax_ref_box.text(0, 0, test, fontsize=REF_FS, fontfamily=BODY_FONT)
            if (t.get_window_extent(renderer=renderer).width / r_bbox.width) <= max_width_ratio: cur = test
            else:
                if cur: c_lines_list.append(cur)
                cur = w
            t.remove()
        if cur: c_lines_list.append(cur)
        all_ref_entries.append((c_lines_list, url_txt)); total_ref_lines += len(c_lines_list) + 1

    step_ref = 0.99 / total_ref_lines; y_ref = 0.995
    for c_lines, url_txt in all_ref_entries:
        for line_str in c_lines:
            ax_ref_box.text(x_start, y_ref, line_str, transform=ax_ref_box.transAxes, fontsize=REF_FS, fontfamily=BODY_FONT, color="white", va="top", ha="left")
            y_ref -= (step_ref * 0.90)
        t_link = ax_ref_box.text(x_start, y_ref, url_txt, transform=ax_ref_box.transAxes, fontsize=REF_FS, fontweight="bold", fontfamily=BODY_FONT, color=LINK_BLUE, va="top", ha="left")
        t_link.set_url(url_txt); y_ref -= (step_ref * 1.05)

render_ref_col(refs_9_expanded, x_start=0.0, max_width_ratio=0.98)

pdf_out = os.path.join(output_dir, "Waterborne_Poster_Final_Revision.pdf")
png_out = os.path.join(output_dir, "Waterborne_Poster_Final_Revision.png")
plt.savefig(png_out, dpi=150)
plt.savefig(pdf_out, dpi=300)
print(f"Part 10: 5ft x 5ft Poster saved successfully at:\\n1. {png_out}\\n2. {pdf_out}")
plt.show()

try:
    from pptx import Presentation
    from pptx.util import Inches
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    slide_layout = prs.slide_layouts[0]
    slide = prs.slides.add_slide(slide_layout)
    title = slide.shapes.title
    subtitle = slide.placeholders[1]
    title.text = "Climate-Related Patterns and Forecasting of Waterborne Disease Burden in Bangladesh (2017–2020)"
    subtitle.text = "Department of Statistics, Begum Rokeya University, Rangpur-5400, Bangladesh"
    pptx_out = os.path.join(output_dir, "Waterborne_Poster_Presentation.pptx")
    prs.save(pptx_out)
    print(f"Presentation PPTX saved successfully at:\\n{pptx_out}")
except ModuleNotFoundError:
    print("python-pptx is not installed. PPTX generation skipped.")
